In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import drive

# Pick models from the GPU's memory: an L4 runs the 26B, an A100 runs both.
gpu_mib = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]
).decode().split()[0])
MODELS = ["gemma4:26b", "gemma4:31b"] if gpu_mib > 30000 else ["gemma4:26b"]

# Models live on the session disk; only results go to Drive.
os.environ["OLLAMA_MODELS"] = "/content/ollama-models"
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

drive.mount('/content/drive')
RUNS_DIR = "/content/drive/MyDrive/dispatcher-runs"
os.makedirs(RUNS_DIR, exist_ok=True)

print(f"GPU memory: {gpu_mib} MiB")
print("Models this session:", MODELS)
!df -h /content

In [ ]:
!curl -fsSL https://ollama.com/install.sh -o install.sh
!apt-get install -y -qq zstd
!sh install.sh
!which ollama

In [ ]:
import subprocess, time

subprocess.run(["pkill", "-f", "ollama serve"])
time.sleep(2)

subprocess.Popen(
    ["ollama", "serve"],
    env=os.environ.copy(),
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

!curl -s http://localhost:11434
print()
!grep -i -E "cuda|compute|gpu" /content/ollama.log | head -5

In [ ]:
for model in MODELS:
    !ollama pull {model}

!ollama list

In [ ]:
if os.path.exists("/content/Dispatcher"):
    os.chdir("/content/Dispatcher")
    !git pull
else:
    !git clone https://github.com/Valientkyton/Dispatcher.git /content/Dispatcher
    os.chdir("/content/Dispatcher")

!pip install -q -r requirements.txt
!git log --oneline -1

In [ ]:
profiles = ["standard", "max"] if "gemma4:31b" in MODELS else ["standard"]

for profile in profiles:
    os.environ["DISPATCHER_PROFILE"] = profile
    print(f"\n===== {profile}: first run (includes loading the model) =====")
    !python -m scripts.smoke_test
    print(f"\n===== {profile}: second run (warm) =====")
    !python -m scripts.smoke_test

!ollama ps

In [ ]:
from datetime import datetime

stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
session_dir = f"{RUNS_DIR}/{stamp}"
os.makedirs(session_dir, exist_ok=True)

!cp /content/ollama.log "{session_dir}/"
if os.path.exists("/content/Dispatcher/traces"):
    !cp -r /content/Dispatcher/traces "{session_dir}/"

!ls -la "{session_dir}"